In [2]:
import os
import json
import torch

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig


In [3]:
BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
TRAIN_PATH = "merged/train.jsonl"
VAL_PATH = "merged/val.jsonl"

ADAPTER_OUTPUT_DIR = "outputs/qwen2.5-coder-7b-pr-review-lora"
MERGED_MODEL_DIR = "outputs/qwen2.5-coder-7b-pr-review-merged"

MAX_SEQ_LENGTH = 512
os.makedirs(ADAPTER_OUTPUT_DIR, exist_ok=True)
os.makedirs(MERGED_MODEL_DIR, exist_ok=True)

device = "cpu"
print("Running on CPU")

Running on CPU


In [4]:


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float32,
    trust_remote_code=True,
)

model.config.use_cache = False 
model.config.pretraining_tp = 1


model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [5]:
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 2,199,552 || all params: 496,232,320 || trainable%: 0.4433


In [6]:
raw_datasets = load_dataset(
    "json",
    data_files={"train": TRAIN_PATH, "validation": VAL_PATH},
)

print(raw_datasets)
print("\nSample record:")
print(json.dumps(raw_datasets["train"][0], indent=2)[:800])


DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 219474
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'metadata'],
        num_rows: 27402
    })
})

Sample record:
{
  "instruction": "You are an experienced software engineer performing a code review. Given the following pull request context and code diff, write a concise, helpful review comment.",
  "input": "Repository: apache/kafka\nLanguage: Java\nPR Title: KAFKA-18628: Deprecate broker.id config (KIP-1232)\nPR Description: Implement [KIP-1232](https://cwiki.apache.org/confluence/x/Hgp3Fw).\n\n- Deprecate the `broker.id` configuration for removal in Kafka 5.0.\n- Log a deprecation warning at broker startup when `broker.id` is\nexplicitly set, pointing users to `node.id`.\n- Pass `node.id` to the tiered storage plugins alongside `broker.id`,\nand switch Kafka's own `RemoteStorageManager` and\n`RemoteLogMetadataManager` imple

In [7]:
def format_example(example):
    """Convert dataset example into Qwen2.5 chat format."""

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert software engineer performing thorough, "
                "constructive code reviews on GitHub Pull Requests."
            ),
        },
        {
            "role": "user",
            "content": f"{example['instruction']}\n\n{example['input']}",
        },
        {
            "role": "assistant",
            "content": example["output"],
        },
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_dataset = raw_datasets["train"].map(
    format_example,
    remove_columns=raw_datasets["train"].column_names,
)

eval_dataset = raw_datasets["validation"].map(
    format_example,
    remove_columns=raw_datasets["validation"].column_names,
)

MAX_TRAIN_SAMPLES = 200
MAX_EVAL_SAMPLES = 50

train_dataset = train_dataset.select(
    range(min(MAX_TRAIN_SAMPLES, len(train_dataset)))
)

eval_dataset = eval_dataset.select(
    range(min(MAX_EVAL_SAMPLES, len(eval_dataset)))
)

print(f"Train examples: {len(train_dataset)}")
print(f"Validation examples: {len(eval_dataset)}")

print("\nSample:\n")
print(train_dataset[0]["text"][:1500])

Train examples: 200
Validation examples: 50

Sample:

<|im_start|>system
You are an expert software engineer performing thorough, constructive code reviews on GitHub Pull Requests.<|im_end|>
<|im_start|>user
You are an experienced software engineer performing a code review. Given the following pull request context and code diff, write a concise, helpful review comment.

Repository: apache/kafka
Language: Java
PR Title: KAFKA-18628: Deprecate broker.id config (KIP-1232)
PR Description: Implement [KIP-1232](https://cwiki.apache.org/confluence/x/Hgp3Fw).

- Deprecate the `broker.id` configuration for removal in Kafka 5.0.
- Log a deprecation warning at broker startup when `broker.id` is
explicitly set, pointing users to `node.id`.
- Pass `node.id` to the tiered storage plugins alongside `broker.id`,
and switch Kafka's own `RemoteStorageManager` and
`RemoteLogMetadataManager` implementations to read it. `broker.id` is
still passed for compatibility and will be dropped in 5.0.
- Reword the 

In [8]:
training_args = SFTConfig(
    output_dir=ADAPTER_OUTPUT_DIR,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,

    num_train_epochs=1,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,

    bf16=False,
    gradient_checkpointing=False,
    optim="adamw_torch",

    max_length=256,
    packing=False,
    dataset_text_field="text",

    logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",

    report_to="none",
    seed=42,

    dataloader_pin_memory=False,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [1]:
import sys
!{sys.executable} -m pip install -U "accelerate>=1.1.0" transformers trl peft bitsandbytes

  Using cached accelerate-1.14.0-py3-none-any.whl (389 kB)
  Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
  Using cached trl-1.9.2-py3-none-any.whl (889 kB)
  Using cached peft-0.20.0-py3-none-any.whl (775 kB)
  Using cached bitsandbytes-0.50.0-py3-none-win_amd64.whl (38.0 MB)


In [9]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.791088,1.988254,1.547437,51200.000000,0.648780


TrainOutput(global_step=50, training_loss=1.5933749294281006, metrics={'train_runtime': 1821.3133, 'train_samples_per_second': 0.11, 'train_steps_per_second': 0.027, 'total_flos': 110622002380800.0, 'train_loss': 1.5933749294281006, 'epoch': 1.0})

In [11]:
trainer.save_model(ADAPTER_OUTPUT_DIR)
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR)

print(f"LoRA adapter saved to: {ADAPTER_OUTPUT_DIR}")


LoRA adapter saved to: outputs/qwen2.5-coder-7b-pr-review-lora


In [12]:
del model, trainer
torch.cuda.empty_cache()

base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

merged_model = PeftModel.from_pretrained(base_model_fp16, ADAPTER_OUTPUT_DIR)
merged_model = merged_model.merge_and_unload()  # folds LoRA deltas into base weights

merged_model.save_pretrained(MERGED_MODEL_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

print(f"Merged model saved to: {MERGED_MODEL_DIR}")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: outputs/qwen2.5-coder-7b-pr-review-merged


In [14]:

inference_tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)

inference_model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_DIR,
    torch_dtype=torch.float32,
)

inference_model.eval()

device = torch.device("cpu")
inference_model.to(device)

print("Inference model loaded successfully on CPU!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Inference model loaded successfully on CPU!


In [15]:
SYSTEM_PROMPT = (
    "You are an expert software engineer performing thorough, constructive "
    "code reviews on GitHub Pull Requests."
)

INSTRUCTION = (
    "Review the following GitHub Pull Request and identify security, "
    "performance, code quality, architecture, maintainability, best-practice, "
    "and documentation issues. Provide constructive review comments."
)

sample_pr_input = """Repository: example-org/example-service
Language: Python
PR Title: Add user password reset endpoint
PR Description: Adds a new POST /reset-password endpoint that lets users request a password reset via email.
File: app/routes/auth.py

Diff:
@@ -10,6 +10,20 @@ def login():
     return jsonify(token=token)

+@app.route("/reset-password", methods=["POST"])
+def reset_password():
+    email = request.json["email"]
+    user = db.query(f"SELECT * FROM users WHERE email = '{email}'")
+    if user:
+        token = str(random.randint(100000, 999999))
+        send_email(email, f"Your reset code is {token}")
+        cache.set(email, token)
+    return jsonify(status="ok")
"""


def generate_review(pr_input, max_new_tokens=400):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{INSTRUCTION}\n\n{pr_input}"},
    ]
    prompt = inference_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(inference_model.device)

    with torch.no_grad():
        output_ids = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.pad_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return inference_tokenizer.decode(generated, skip_special_tokens=True)


review = generate_review(sample_pr_input)
print("=== Generated Review ===\n")
print(review)


=== Generated Review ===

Reviewer: JohnDoe <john.doe@example.com>
File: app/routes/auth.py
Lines diff:
@@ -8,7 +8,7 @@
 from flask import jsonify, request
 from .models import User
 from .utils import send_email, get_user_by_email
-from .config import SECRET_KEY
+from .config import SECRET_KEY, EMAIL_SENDER, EMAIL_HOST, EMAIL_PORT, EMAIL_USE_TLS
 from .database import db
@@ -34,6 +34,15 @@
 @app.route("/login", methods=["GET"])
 def login():
     return jsonify(token=token)
+
+@app.route("/reset-password", methods=["POST"])
+def reset_password():
+    email = request.json["email"]
+    user = db.query(f"SELECT * FROM users WHERE email = '{email}'")
+    if user:
+        token = str(random.randint(100000, 999999))
+        send_email(email, f"Your reset code is {token}")
+        cache.set(email, token)
+    return jsonify(status="ok")
+
 @app.route("/logout", methods=["DELETE"])
 def logout():
     session.pop("user_id", None)
     return jsonify()
```

Security Analysis:

There are 